# Capture spike — does the sniffer see the chat this PC sends to itself?

The app reads the game's chat off the wire with a raw socket on one adapter's address. To test that **end to end with no game**, this PC plays the
game server: a TCP server on port 5003 sends real chat frames to a client on the same PC, and the app (sniffer on, `network_interface` set) must
publish them as `packet-event`s. Windows may deliver a PC's traffic to itself without it ever passing the adapter, so it is tried on the **LAN address**
and on **127.0.0.1**; each is a row, and the evidence says how many of the lines arrived. Needs a `test-env` exe, Node 20+ and Rust (`cargo`, to build the
frames); run Jupyter as Administrator. The app starts its sniffer only if the exe has its **firewall rule**: set `ADD_RULE = True` to let this notebook make
one for the copy it starts (this copy only, removed at the end) -- it edits the Windows firewall, so it is off by default. The same pipeline without Jupyter:
`python -m runbook.run capture-spike [--add-firewall-rule]`.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import bridge, common
from runbook.common import Recorder
from runbook.pipelines.capture_spike import CaptureSpike
rec = Recorder("capture-spike")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")
bridge.ensure_installed()                # npm ci in runbook/bridge, the first time

In [ ]:
EXE = common.ask_exe()          # target/release/resonance-stream.exe of this checkout, unless RUNBOOK_TEXT_LOCAL_EXE says otherwise
ADD_RULE = False   # True: make (and afterwards remove) a firewall rule for each copy the pipeline starts
CaptureSpike(rec, EXE, common.RUNS, add_firewall_rule=ADD_RULE).run()

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())